# From ODS to DWD: versions and `known_at`, in SQL

ODS holds every capture of every file, unchanged in meaning. DWD turns that into **facts with the time they became known**: one row per version of a value, its slot in JST, its value in MW, its `known_at` and the `basis` for it. Every feature then reads DWD through `as_of(T)`: per key, the latest version with `known_at ≤ T`. This is where point-in-time correctness is either built or lost.

Notebook 02 set the rules in pandas, on one day's captures. This notebook writes them in **BigQuery SQL over the whole ODS tables**, the way the dbt models will, for the tables the solar forecast reads: TEPCO's actuals (the target and its lags), the three weather tables, and TEPCO's next-day forecast (the benchmark). Capacity has its own notebook (03); section 6 brings its `known_at` into SQL.

| Question | Where |
| --- | --- |
| Where does each source's capture start, the point where `known_at` switches from rule to capture? | Section 1 |
| How do captures of the target become versions, and what happens to data loaded after capture start? | Section 2 |
| Which weather forecast can a forecast at `T` use, and does every target day have one? | Section 3 |
| How do hourly weather values map onto 30-minute slots? | Section 3.5 |
| How does the benchmark fit in? | Section 4 |
| What shape should `as_of` take to serve every decision time of a backtest at once? Does it hold against future rows? | Section 5 |
| When did each capacity release become known, in SQL, and what do daily copies of the page change? | Section 6 |

`T` is 09:00 JST on D−1 throughout. Claims are labelled **Observed** (a cell shows it) or **Source** (quoted, with a link). [What this exploration made known](#What-this-exploration-made-known) sums it up, with what is still open.

## 0. Setup

**Step.** Everything here is SQL against the ODS tables in dev BigQuery, written the way the dbt models will be: the queries in this notebook are the first drafts of DWD. `sql()` runs one with named parameters and prints how much it scanned, since the `as_of` design has to stay cheap when it runs for every decision time at once.

In [1]:
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
from google.cloud import bigquery

pd.set_option("display.width", 160, "display.max_columns", 30, "display.max_colwidth", 60)


def bigquery_project() -> str:
    if project := os.environ.get("JTF_BIGQUERY_PROJECT"):
        return project
    lines = Path("../.env").read_text().splitlines()
    env = dict(line.split("=", 1) for line in lines if "=" in line and not line.startswith("#"))
    return env["JTF_BIGQUERY_PROJECT"]


PROJECT = bigquery_project()
bq = bigquery.Client(project=PROJECT)


def _parameter(name, value):
    if isinstance(value, (pd.Timestamp, datetime)):
        return bigquery.ScalarQueryParameter(name, "TIMESTAMP", pd.Timestamp(value).to_pydatetime())
    if isinstance(value, int):
        return bigquery.ScalarQueryParameter(name, "INT64", value)
    return bigquery.ScalarQueryParameter(name, "STRING", value)


def sql(query: str, **params) -> pd.DataFrame:
    """Run a query against the dev ODS dataset; print how much it scanned."""
    config = bigquery.QueryJobConfig(
        default_dataset=f"{PROJECT}.ods",
        use_query_cache=False,  # so the scanned size is the real cost
        query_parameters=[_parameter(k, v) for k, v in params.items()],
    )
    job = bq.query(query, job_config=config)
    rows = job.to_dataframe(create_bqstorage_client=False)
    print(f"[{(job.total_bytes_processed or 0) / 1e6:.1f} MB scanned]")
    return rows


print("ODS dataset:", f"{PROJECT}.ods")

ODS dataset: jepx-tokyo-forecast-dev.ods


## 1. Capture start, per source

**Step.** The `known_at` rule (notebook 02, section 9) switches at each source's **capture start**: its first `daily` capture. Before it, a history rule stands in; after it, the capture that first showed a value is the evidence. The manifest `raw_files` has one row per parsed file, so capture start is one query, and DWD will read it from there rather than from a setting.

In [2]:
capture_start = sql("""
    SELECT source, MIN(IF(load_type = 'daily', fetched_at, NULL)) AS capture_start,
           COUNTIF(load_type = 'daily') AS daily_files, COUNTIF(load_type != 'daily') AS other_files
    FROM raw_files
    GROUP BY source
    ORDER BY source
""")
capture_start["capture_start_jst"] = capture_start["capture_start"].dt.tz_convert("Asia/Tokyo")
capture_start[["source", "capture_start_jst", "daily_files", "other_files"]]

[0.1 MB scanned]


,source,capture_start_jst,daily_files,other_files
0,jepx_spot,2026-11-15 11:08:12.892261+09:00,4,10
1,meti_fit_capacity,2026-11-17 08:45:19.719609+09:00,1,119
2,openmeteo_live,2026-11-15 11:36:27.058501+09:00,4,0
3,openmeteo_previous_runs,NaT,0,36
4,openmeteo_single_runs,NaT,0,1362
5,tepco_actuals,2026-11-15 11:04:16.643393+09:00,10,42
6,tepco_area_actuals_today,2026-11-15 12:24:01.519817+09:00,3,0
7,tepco_area_bg_plan,2026-11-15 12:24:01.524178+09:00,3,0
8,tepco_area_bg_plan_next_day,2026-11-15 12:24:01.521792+09:00,5,0
9,tepco_area_forecast,2026-11-15 12:24:01.519534+09:00,3,0


**What it shows.** **Observed:**

- The daily sources started on 2026-11-15 between 11:04 and 12:24 JST, the first captures of capture day (test runs included: they are real captures, the earliest evidence we have)
- `openmeteo_previous_runs` and `openmeteo_single_runs` have **no capture start**: they are history-only sources, so every one of their values is dated by its rule
- `meti_fit_capacity` started on 2026-11-17 at 08:45, the day after its 33 releases were loaded, so all of them are dated by notebook 03's rule (section 6)

**Learned → design:** capture start is data, not a setting: DWD reads it from `ods.raw_files`, per source. A source added later (capacity) gets its own capture start, and its history keeps the rule.

## 2. The target: TEPCO actuals as versions

### 2.1 From captures to versions

**Step.** One query does what notebook 02 did in pandas, on every row of `ods.tepco_actuals`:

1. Read the newest `parser_version` of each file (a re-parse appends; it never replaces)
2. Turn `date` and `time` (JST labels, slot start) into a `slot_start` timestamp; the values are already MW averaged over the slot
3. Unpivot the measures the solar path needs: `solar`, `solar_curtailed`, and `wind` (TEPCO's benchmark forecast is wind and solar together)
4. Order each (slot, measure) by capture time and keep a row only where the value differs from the previous capture's: a **version**
5. Stamp `known_at`: a first version whose rule time (slot end + 1 hour) falls before capture start takes the rule; every other version takes the capture that first showed it

`VERSIONS` is kept as a string: later sections build on it, as dbt models will build on `dwd_tepco_actuals`.

In [3]:
VERSIONS = """
WITH capture_start AS (
  SELECT MIN(fetched_at) AS t FROM raw_files
  WHERE source = 'tepco_actuals' AND load_type = 'daily'
),
newest_parse AS (
  SELECT * FROM tepco_actuals
  QUALIFY parser_version = MAX(parser_version) OVER (PARTITION BY source_file)
),
observed AS (
  SELECT
    TIMESTAMP(DATETIME(PARSE_DATE('%Y/%m/%d', date), PARSE_TIME('%H:%M', time)), 'Asia/Tokyo')
      AS slot_start,
    measure, value_mw, fetched_at, load_type, source_file
  FROM newest_parse
  UNPIVOT INCLUDE NULLS (value_mw FOR measure IN (solar, solar_curtailed, wind))
),
changes AS (
  SELECT *, LAG(value_mw) OVER w AS previous, ROW_NUMBER() OVER w AS sighting
  FROM observed
  WINDOW w AS (PARTITION BY slot_start, measure ORDER BY fetched_at, source_file)
),
versions AS (
  SELECT
    slot_start, measure, value_mw, previous, sighting, load_type, source_file,
    fetched_at AS first_seen,
    TIMESTAMP_ADD(slot_start, INTERVAL 90 MINUTE) AS rule_time,
    sighting = 1
      AND TIMESTAMP_ADD(slot_start, INTERVAL 90 MINUTE) < (SELECT t FROM capture_start) AS by_rule
  FROM changes
  WHERE sighting = 1 OR value_mw IS DISTINCT FROM previous
)
SELECT
  slot_start, measure, value_mw, load_type, source_file, first_seen, rule_time,
  IF(by_rule, rule_time, first_seen) AS known_at,
  IF(by_rule, 'rule', 'captured') AS basis,
  sighting > 1 AS correction
FROM versions
"""

sql(f"""
    SELECT basis, load_type AS first_seen_in, correction, COUNT(*) AS versions,
           COUNT(DISTINCT slot_start) AS slots,
           MIN(slot_start) AS first_slot, MAX(slot_start) AS last_slot
    FROM ({VERSIONS})
    GROUP BY 1, 2, 3
    ORDER BY 1 DESC, 2
""")

[7.5 MB scanned]


,basis,first_seen_in,correction,versions,slots,first_slot,last_slot
0,rule,daily,False,2076,692,2026-10-31 15:00:00+00:00,2026-11-15 00:30:00+00:00
1,rule,initial,False,144576,48192,2024-01-31 15:00:00+00:00,2026-10-31 14:30:00+00:00
2,captured,daily,False,270,90,2026-11-15 01:00:00+00:00,2026-11-16 23:00:00+00:00
3,captured,recapture,False,9,3,2026-11-15 03:00:00+00:00,2026-11-15 04:00:00+00:00


**What it shows.** **Observed:** 146,931 versions over 48,977 slots and three measures, in one query scanning 7.5 MB:

- 144,576 versions from the initial load (February 2024 to October 2026) and 2,076 from the first daily capture (1–15 November up to 09:30 JST): history, dated by the rule (slot end + 1 hour)
- 270 versions first seen by a daily capture after capture start (10:00 on 15 November to 08:00 on the 17th), and 9 first seen by the 14:02 re-capture on 15 November: dated by the capture
- **No corrections**: not one value changed from one capture to the next

**Learned → design:** versions and `known_at` are one window function and a `CASE`, cheap enough to rebuild from ODS on every run. `UNPIVOT INCLUDE NULLS` keeps a value that is null in one capture and present in the next as a version of its own ([BigQuery `UNPIVOT`](https://cloud.google.com/bigquery/docs/reference/standard-sql/query-syntax#unpivot_operator)).

**Step: does every slot have exactly one version per measure?** Count the captures that saw each (slot, measure) and the versions they produced. A slot seen many times with one version means TEPCO never changed it while we watched.

In [4]:
sql(f"""
    WITH v AS ({VERSIONS}),
    seen AS (
      SELECT
        TIMESTAMP(DATETIME(PARSE_DATE('%Y/%m/%d', date), PARSE_TIME('%H:%M', time)), 'Asia/Tokyo')
          AS slot_start,
        COUNT(DISTINCT source_file) AS captures
      FROM tepco_actuals
      GROUP BY 1
    ),
    solar AS (SELECT slot_start, COUNT(*) AS versions FROM v WHERE measure = 'solar' GROUP BY 1)
    SELECT captures, versions AS solar_versions, COUNT(*) AS slots
    FROM seen JOIN solar USING (slot_start)
    GROUP BY captures, versions
    ORDER BY captures, versions
""")

[6.9 MB scanned]


,captures,solar_versions,slots
0,1,1,46752
1,2,1,38
2,3,1,1491
3,4,1,1
4,5,1,1
5,7,1,1
6,9,1,693


**What it shows.** **Observed:** 46,752 slots were seen in one capture, 1,491 in three (October's whole file, captured on each of the three days), 693 in nine (November's first two weeks, in every capture from 15 to 17 November), and every slot has exactly one solar version. Three days of watching found no change.

**Learned → design:** corrections are rare and late: notebook 01 found files rewritten weeks after their month. The versions logic costs nothing while there are none; the monthly re-capture is what will bring them in.

### 2.2 The seam at capture start, and a backfill that came late

**Step: look at the slots around capture start**, where the rule hands over to the captures. One row per slot (solar only), with the rule's time, the capture that first showed the slot, and the `known_at` chosen. The interesting rows are those first seen by a capture *other than* the daily one: anything loaded after capture start by a history load or a re-capture must take that load's time, never the rule, or it would be dated earlier than anyone could have had it.

In [5]:
seam = sql(f"""
    SELECT slot_start, load_type AS first_seen_in, first_seen, rule_time, known_at, basis
    FROM ({VERSIONS})
    WHERE measure = 'solar'
      AND slot_start BETWEEN TIMESTAMP('2026-11-15 08:30:00', 'Asia/Tokyo')
                         AND TIMESTAMP('2026-11-15 13:30:00', 'Asia/Tokyo')
    ORDER BY slot_start
""")
for column in ("slot_start", "first_seen", "rule_time", "known_at"):
    seam[column] = seam[column].dt.tz_convert("Asia/Tokyo").dt.strftime("%m-%d %H:%M")
seam

[7.5 MB scanned]


,slot_start,first_seen_in,first_seen,rule_time,known_at,basis
0,11-15 08:30,daily,11-15 11:04,11-15 10:00,11-15 10:00,rule
1,11-15 09:00,daily,11-15 11:04,11-15 10:30,11-15 10:30,rule
2,11-15 09:30,daily,11-15 11:04,11-15 11:00,11-15 11:00,rule
3,11-15 10:00,daily,11-15 11:04,11-15 11:30,11-15 11:04,captured
4,11-15 10:30,daily,11-15 11:11,11-15 12:00,11-15 11:11,captured
5,11-15 11:00,daily,11-15 11:36,11-15 12:30,11-15 11:36,captured
6,11-15 11:30,daily,11-15 12:24,11-15 13:00,11-15 12:24,captured
7,11-15 12:00,recapture,11-15 14:02,11-15 13:30,11-15 14:02,captured
8,11-15 12:30,recapture,11-15 14:02,11-15 14:00,11-15 14:02,captured
9,11-15 13:00,recapture,11-15 14:02,11-15 14:30,11-15 14:02,captured


**What it shows.** **Observed:** the hand-over, and a backfill that came late:

- Up to the 09:30 slot the rule's time (slot end + 1 hour) is before capture start (11:04), so the rule stands. From the 10:00 slot on, the capture that first showed a slot dates it
- The 12:00, 12:30 and 13:00 slots were first seen by the **re-capture at 14:02**, after capture start. They take 14:02. The rule would have said 13:30 and 14:00 for the first two: **earlier than we had them**. That is the backfill leak, in real data, and rule 1 (the rule only before capture start) is what prevents it
- The 13:30 slot was first seen by the next morning's capture (08:45 on 16 November). TEPCO most likely published it soon after 14:00 on 15 November, but we hold no capture showing it, so `known_at` errs late, never early

**Learned → design:** the dbt unit test for a late backfill can use exactly this case: three slots first seen by a non-daily load after capture start must carry that load's time.

### 2.3 The long history: yearly files, hourly

**Step: where the yearly files end and the monthly ones begin.** The yearly `area-YYYY.csv` files are hourly in 万kWh (10 MW averaged over the hour); the monthly files are 30-minute MW. The training window starts on target day 2024-02-17 (notebook 02), and its lags reach back to 2024-02-08 at most.

In [6]:
sql("""
    SELECT 'yearly (hourly, 万kWh)' AS files,
           MIN(PARSE_DATE('%Y/%m/%d', date)) AS first_day,
           MAX(PARSE_DATE('%Y/%m/%d', date)) AS last_day,
           COUNT(*) AS row_count
    FROM tepco_actuals_yearly
    UNION ALL
    SELECT 'monthly (30 min, MW)',
           MIN(PARSE_DATE('%Y/%m/%d', date)), MAX(PARSE_DATE('%Y/%m/%d', date)),
           COUNT(*)
    FROM tepco_actuals
""")

[1.4 MB scanned]


,files,first_day,last_day,row_count
0,"yearly (hourly, 万kWh)",2016-04-01,2024-01-31,68688
1,"monthly (30 min, MW)",2024-02-01,2026-11-17,57554


**What it shows.** **Observed:** the yearly files run from 2016-04-01 to 2024-01-31 (68,688 hourly rows), the monthly files from 2024-02-01 to today: they meet without overlap. The training window starts on target day 2024-02-17, and its longest lag (the last seven complete days, D−8 to D−2) reaches back to 2024-02-09.

**Learned → design:** the target and every lag come from the monthly 30-minute files alone, so no stitching across grains is needed for phase 1. The yearly files get their own hourly DWD model for anything longer (a climatology, say), and that model must allow for the step in TEPCO's solar in early 2021 (notebook 03, section 6).

## 3. The weather: what a forecast at `T` can use

`T` is 09:00 JST on D−1 (00:00 UTC on D−1). Each weather table answers "when was this forecast known" differently (notebook 02, section 9): Previous Runs by a rule on the lead, Single Runs by the run's start plus the model's delay, the live capture by its fetch time. Open-Meteo stamps hours in UTC (asserted in ODS), and radiation at hour `h` is the mean of the hour before `h` ([Open-Meteo docs](https://open-meteo.com/en/docs)).

### 3.1 Previous Runs: which lead is usable at `T`?

**Step.** For every daylight hour of every target day (hours where MSM's lead-0 radiation is above zero, at any point), compute when each lead was known, `valid − 24 h × lead + delay` (MSM 3 h), and whether that is at or before `T`. The share of daylight hours known at `T` says which lead a feature may use.

In [7]:
sql("""
    WITH daylight AS (
      SELECT DISTINCT valid_time FROM openmeteo_previous_runs
      WHERE model = 'jma_msm' AND lead_days = 0 AND shortwave_radiation > 0
    ),
    hours AS (
      SELECT valid_time, lead,
        TIMESTAMP_ADD(TIMESTAMP_SUB(valid_time, INTERVAL 24 * lead HOUR), INTERVAL 3 HOUR)
          AS known_at,
        TIMESTAMP(DATETIME(DATE_SUB(DATE(valid_time, 'Asia/Tokyo'), INTERVAL 1 DAY), TIME(9, 0, 0)),
                  'Asia/Tokyo') AS t
      FROM daylight, UNNEST([0, 1, 2]) AS lead
    )
    SELECT lead AS lead_days,
           ROUND(100 * COUNTIF(known_at <= t) / COUNT(*), 1) AS pct_daylight_hours_known_at_t,
           MAX(EXTRACT(HOUR FROM valid_time AT TIME ZONE 'Asia/Tokyo')) AS latest_daylight_hour_jst,
           MAX(IF(known_at <= t, EXTRACT(HOUR FROM valid_time AT TIME ZONE 'Asia/Tokyo'), NULL))
             AS latest_hour_known_jst
    FROM hours
    GROUP BY lead
    ORDER BY lead
""")

[65.1 MB scanned]


,lead_days,pct_daylight_hours_known_at_t,latest_daylight_hour_jst,latest_hour_known_jst
0,0,0.0,19,<NA>
1,1,6.9,19,6
2,2,100.0,19,19


**What it shows.** **Observed:** at `T`, lead 2 (`previous_day2`) covers 100% of D's daylight hours; lead 1 covers 6.9%, only the hours up to 06:00 JST; lead 0 covers none. The midday hours, which carry most of the solar energy, are never covered by lead 1.

**Learned → design:** features use lead 2 only. Leads 0 and 1 stay in DWD for the leak scenario F2, where a feature built on them shows how much better a forecast looks with information it couldn't have had.

### 3.2 Single Runs: the newest run at `T`, and how far it reaches

**Step.** For each target day in the archive and each model, find the newest run known at `T` (`run_time` + assumed delay: MSM 3 h, GSM and ECMWF 6 h) and how much of D's daylight it covers. The archive is complete since 16 November (section 3.4): target days from 14 May, two days after JMA's archive starts (notebook 01), to 16 November, the last its newest runs serve.

In [8]:
single = sql("""
    WITH runs AS (
      SELECT model, run_time, MAX(valid_time) AS last_valid,
        TIMESTAMP_ADD(run_time, INTERVAL IF(model = 'jma_msm', 3, 6) HOUR) AS known_at
      FROM openmeteo_single_runs
      GROUP BY model, run_time
    ),
    days AS (
      SELECT d, TIMESTAMP(DATETIME(DATE_SUB(d, INTERVAL 1 DAY), TIME(9, 0, 0)), 'Asia/Tokyo') AS t
      FROM UNNEST(GENERATE_DATE_ARRAY('2026-05-14', '2026-11-16')) AS d
    ),
    newest AS (
      SELECT d, t, model, run_time, known_at, last_valid
      FROM days JOIN runs ON runs.known_at <= days.t
      QUALIFY ROW_NUMBER() OVER (PARTITION BY d, model ORDER BY run_time DESC) = 1
    ),
    daylight AS (
      SELECT DATE(valid_time, 'Asia/Tokyo') AS d, MAX(valid_time) AS last_light
      FROM openmeteo_previous_runs
      WHERE model = 'jma_msm' AND lead_days = 0 AND shortwave_radiation > 0
      GROUP BY 1
    )
    SELECT model,
           COUNT(*) AS target_days,
           COUNTIF(known_at = t) AS run_known_exactly_at_t,
           ROUND(AVG(TIMESTAMP_DIFF(t, run_time, MINUTE)) / 60, 1) AS run_age_at_t_h,
           COUNTIF(last_valid >= last_light) AS covers_all_daylight,
           MIN(TIMESTAMP_DIFF(last_valid, last_light, HOUR)) AS min_hours_past_last_daylight
    FROM newest JOIN daylight USING (d)
    GROUP BY model
    ORDER BY model
""")
single

[100.1 MB scanned]


,model,target_days,run_known_exactly_at_t,run_age_at_t_h,covers_all_daylight,min_hours_past_last_daylight
0,ecmwf_ifs025,187,186,6.1,187,82
1,jma_gsm,187,186,6.1,187,75
2,jma_msm,187,186,3.1,186,-20


**What it shows.** **Observed:** over 187 target days, 14 May to 16 November:

- The newest run known at `T` is MSM's 21 UTC run (3 hours old at `T`), GSM's and ECMWF's 18 UTC runs (6 hours old)
- With the assumed delays (MSM 3 h, GSM and ECMWF 6 h), **that newest run becomes available exactly at `T`** on 186 of 187 days for each model
- The exceptions are days after missing runs (recorded gaps): the newest run is then days old. On 12 June ECMWF's is from 9 June, and on 9 November GSM's and MSM's are from 6 November. GSM and ECMWF still cover D, but **MSM's ends at 21:00 JST on 8 November, before D begins**: its 39-hour range can't bridge two missing days
- On every other day, the chosen run covers all of D's daylight; MSM's ends 2 hours after the last daylight hour (its range ends at 21:00 JST on D)

**Learned → design:** with `known_at ≤ T`, a delay assumption off by a single minute changes which run *every* day uses. That is too fragile to trust before the delays are measured. Until they are, DWS uses the older of the two runs captured per model and day (notebook 01 captured both for this reason), and the newest only once measured delays put it safely before `T`. Either way, DWS takes a run only if it covers D's daylight, and falls back to another source when none does, as on 9 November for MSM.

### 3.3 The live capture

**Step.** The 08:45 capture is known at its fetch time, before `T`. Which run it holds the response doesn't say (notebook 02, section 6), so its `known_at` is the fetch, the run unknown. What matters is that it covers D: compare each capture's last valid hour with the end of D in UTC.

In [9]:
live = sql("""
    SELECT fetched_at, MIN(valid_time) AS first_valid, MAX(valid_time) AS last_valid,
           COUNT(*) AS row_count
    FROM openmeteo_live
    GROUP BY fetched_at
    ORDER BY fetched_at
""")
live["fetched_jst"] = live["fetched_at"].dt.tz_convert("Asia/Tokyo")
live["target_day_D"] = (live["fetched_jst"] + pd.Timedelta(days=1)).dt.date
live["end_of_D_utc"] = [
    pd.Timestamp(d, tz="Asia/Tokyo") + pd.Timedelta(days=1) - pd.Timedelta(hours=1)
    for d in live["target_day_D"]
]
live["covers_D"] = live["last_valid"] >= live["end_of_D_utc"].dt.tz_convert("UTC")
live["before_T"] = live["fetched_jst"].dt.hour * 60 + live["fetched_jst"].dt.minute <= 9 * 60
live[["fetched_jst", "before_T", "target_day_D", "last_valid", "covers_D", "row_count"]]

[0.1 MB scanned]


,fetched_jst,before_T,target_day_D,last_valid,covers_D,row_count
0,2026-11-15 11:36:27.058501+09:00,False,2026-11-16,2026-11-17 23:00:00+00:00,True,1899
1,2026-11-15 12:24:02.048853+09:00,False,2026-11-16,2026-11-17 23:00:00+00:00,True,1899
2,2026-11-16 08:45:18.981002+09:00,True,2026-11-17,2026-11-17 23:00:00+00:00,True,1944
3,2026-11-17 08:46:06.982063+09:00,True,2026-11-18,2026-11-18 23:00:00+00:00,True,1944


**What it shows.** **Observed:** the live captures of 16 and 17 November at 08:45 JST, before `T`, each cover their target day (17 and 18 November) to the end: Open-Meteo's three forecast days start at 00:00 UTC on the day of the request. The two captures of 15 November were test runs at 11:36 and 12:24, after `T`, so they serve no target day.

**Learned → design:** a live capture serves D only if it was fetched before `T` on D−1. Its `known_at` is the fetch, and which run it holds is unknown: the response doesn't say.

### 3.4 The seam: which weather each target day can use

**Step.** Around capture start, list per target day what is available at `T`: Previous Runs at lead 2 (how many of D's 24 hours, from MSM), the newest Single Run, and a live capture fetched by `T` that covers D. Previous Runs and Single Runs are history-only sources. Both were loaded on 2026-11-15; the first run of this section found the gap described below, so on 2026-11-16 Previous Runs was topped up and Single Runs resumed once Open-Meteo's daily quota allowed. The live capture's first capture before `T` was at 08:45 on 2026-11-16.

In [10]:
sql("""
    WITH days AS (
      SELECT d, TIMESTAMP(DATETIME(DATE_SUB(d, INTERVAL 1 DAY), TIME(9, 0, 0)), 'Asia/Tokyo') AS t
      FROM UNNEST(GENERATE_DATE_ARRAY('2026-11-12', '2026-11-17')) AS d
    ),
    prev2 AS (
      SELECT DATE(valid_time, 'Asia/Tokyo') AS d, COUNT(DISTINCT valid_time) AS hours
      FROM openmeteo_previous_runs
      WHERE model = 'jma_msm' AND lead_days = 2
      GROUP BY 1
    ),
    live AS (
      SELECT fetched_at, MAX(valid_time) AS last_valid FROM openmeteo_live GROUP BY fetched_at
    ),
    runs AS (
      SELECT DISTINCT TIMESTAMP_ADD(run_time, INTERVAL IF(model = 'jma_msm', 3, 6) HOUR) AS known_at
      FROM openmeteo_single_runs
    )
    SELECT d AS target_day,
           IFNULL(prev2.hours, 0) AS previous_day2_hours_of_24,
           EXISTS (SELECT 1 FROM runs WHERE runs.known_at <= t
                   AND runs.known_at > TIMESTAMP_SUB(t, INTERVAL 1 DAY)) AS single_run_at_t,
           EXISTS (SELECT 1 FROM live WHERE live.fetched_at <= t
                   AND live.last_valid >= TIMESTAMP(DATETIME(d, TIME(23, 0, 0)), 'Asia/Tokyo'))
             AS live_at_t
    FROM days LEFT JOIN prev2 USING (d)
    ORDER BY d
""")

[79.5 MB scanned]


,target_day,previous_day2_hours_of_24,single_run_at_t,live_at_t
0,2026-11-12,24,True,False
1,2026-11-13,24,True,False
2,2026-11-14,24,True,False
3,2026-11-15,24,True,False
4,2026-11-16,9,True,False
5,2026-11-17,0,False,True


**What it shows.** **Observed:** after the top-ups, every target day around capture start has weather known at `T`, but no single source spans the seam:

| Target day | Weather known at `T` |
| --- | --- |
| Up to 2026-11-15 | Previous Runs, lead 2: all 24 hours; Single Runs from all three models |
| 2026-11-16 | Single Runs from all three models; lead 2 for only 9 hours, up to 08:00 JST |
| 2026-11-17 | The live capture only |

Before the top-ups, 15 November had 9 hours and 16 November nothing. A Previous Runs history stops at the end of the day before its load, in UTC: loaded on 16 November, it reaches 08:00 JST on the 16th. The Single Runs archive ends with the runs known at 09:00 on 15 November, exactly `T` for the 16th. The live capture serves from the 17th.

**Learned → design:** a history-only source stops where its load stopped, and a live capture serves only from the day after it starts, so a seam at capture start is built in. Topping the histories up after capture start closes it, but each side of it is served by a different source. The weather DWS therefore picks a source per target day (lead 2, a Single Run that covers D, the live capture), and a dbt data test checks that every target day of the window has weather known at `T`.

**Step: the same history, captured twice.** The top-up of Previous Runs on 16 November fetched 1–15 November, and the history load of the 15th had fetched 1–14 November, so those days are held twice. Notebook 02 (section 9) expected an archive fetched once, with no later versions. For every (point, model, lead, hour) held in more than one file: do the copies agree?

In [11]:
sql("""
    WITH seen AS (
      SELECT point, model, lead_days, valid_time, source_file,
        TO_JSON_STRING(STRUCT(shortwave_radiation, direct_radiation, diffuse_radiation,
                              cloud_cover, temperature_2m, wind_speed_10m)) AS v
      FROM openmeteo_previous_runs
    ),
    held_twice AS (
      SELECT model, lead_days, valid_time, point,
             COUNT(DISTINCT source_file) AS files, COUNT(DISTINCT v) AS distinct_values
      FROM seen
      GROUP BY model, lead_days, valid_time, point
      HAVING files > 1
    )
    SELECT model, lead_days,
           COUNT(*) AS hours_held_twice,
           COUNTIF(distinct_values > 1) AS hours_that_differ,
           FORMAT_TIMESTAMP('%Y-%m-%d %H:%M', MIN(IF(distinct_values > 1, valid_time, NULL)),
                            'Asia/Tokyo') AS first_differing_jst,
           FORMAT_TIMESTAMP('%Y-%m-%d %H:%M', MAX(IF(distinct_values > 1, valid_time, NULL)),
                            'Asia/Tokyo') AS last_differing_jst
    FROM held_twice
    GROUP BY model, lead_days
    ORDER BY model, lead_days
""")

[344.6 MB scanned]


,model,lead_days,hours_held_twice,hours_that_differ,first_differing_jst,last_differing_jst
0,ecmwf_ifs025,0,3024,33,2026-11-15 04:00,2026-11-15 08:00
1,ecmwf_ifs025,1,3024,0,NaN,NaN
2,ecmwf_ifs025,2,3024,0,NaN,NaN
3,jma_gsm,0,3024,71,2026-11-14 22:00,2026-11-15 08:00
4,jma_gsm,1,3024,0,NaN,NaN
5,jma_gsm,2,3024,0,NaN,NaN
6,jma_msm,0,3024,0,NaN,NaN
7,jma_msm,1,3024,0,NaN,NaN
8,jma_msm,2,2871,0,NaN,NaN


**What it shows.** **Observed:** 27,063 (point, model, lead, hour) rows are held twice: every one of 1–14 November that both files cover. Leads 1 and 2 agree everywhere, and MSM agrees at every lead. **104 differ, all at lead 0**: 71 of GSM from 22:00 on 14 November and 33 of ECMWF from 04:00 on 15 November, up to 08:00 on the 15th. Those are the last hours of the first file, fetched at 14:01 that day; lead 0 is the newest forecast for an hour, and for the hours just before that fetch newer runs were still replacing it.

**Learned → design:** a later fetch of the archive changes only what a later run can still reach: lead 0 of the most recent hours. No special case is needed: DWD versions Previous Runs like every other source, so a copy seen again adds nothing, and a changed value becomes a second version, known at the later fetch. Lead 2, which the features read, never changed. Reading ODS directly counts this history twice, which is why section 3.5 keeps one capture per hour and features read DWD.

### 3.5 Hourly weather onto 30-minute slots

**Step: one point, one morning.** Radiation stamped at hour `h` is the mean over the hour before `h`, so both 30-minute slots inside that hour take it: slot `h−1:00` and slot `h−0:30`. Temperature and cloud cover are instantaneous at `h`, so a slot takes the value interpolated to its middle. Getting this backwards shifts radiation by an hour: a silent lag of the kind F4 is about.

In [12]:
sql("""
    WITH hourly AS (
      SELECT valid_time, shortwave_radiation AS ghi, temperature_2m AS temp
      FROM openmeteo_previous_runs
      WHERE point = 'tokyo' AND model = 'jma_msm' AND lead_days = 2
        AND valid_time BETWEEN TIMESTAMP('2026-11-10 06:00:00', 'Asia/Tokyo')
                           AND TIMESTAMP('2026-11-10 09:00:00', 'Asia/Tokyo')
      -- one row per hour: this history is held twice (section 3.4); DWD keeps versions instead
      QUALIFY ROW_NUMBER() OVER (PARTITION BY valid_time ORDER BY fetched_at) = 1
    ),
    slots AS (
      SELECT TIMESTAMP_ADD(TIMESTAMP('2026-11-10 06:00:00', 'Asia/Tokyo'), INTERVAL 30 * i MINUTE)
        AS slot_start
      FROM UNNEST(GENERATE_ARRAY(0, 5)) AS i
    )
    , mid AS (SELECT slot_start, TIMESTAMP_ADD(slot_start, INTERVAL 15 MINUTE) AS m FROM slots)
    SELECT
      FORMAT_TIMESTAMP('%H:%M', slot_start, 'Asia/Tokyo') AS slot_jst,
      -- radiation: the stamp at the end of the hour that contains the slot
      FORMAT_TIMESTAMP('%H:%M', ending.valid_time, 'Asia/Tokyo') AS ghi_stamp_jst,
      ending.ghi AS ghi_wm2,
      -- temperature: interpolated linearly to the slot's middle
      ROUND(a.temp + (b.temp - a.temp) * EXTRACT(MINUTE FROM m) / 60, 2) AS temp_mid_slot,
      a.temp AS temp_at_hour_before, b.temp AS temp_at_hour_after
    FROM mid
    LEFT JOIN hourly AS ending
      ON ending.valid_time = TIMESTAMP_ADD(TIMESTAMP_TRUNC(slot_start, HOUR), INTERVAL 1 HOUR)
    LEFT JOIN hourly AS a ON a.valid_time = TIMESTAMP_TRUNC(m, HOUR)
    LEFT JOIN hourly AS b ON b.valid_time = TIMESTAMP_ADD(TIMESTAMP_TRUNC(m, HOUR), INTERVAL 1 HOUR)
    ORDER BY slot_start
""")

[8.4 MB scanned]


,slot_jst,ghi_stamp_jst,ghi_wm2,temp_mid_slot,temp_at_hour_before,temp_at_hour_after
0,06:00,07:00,5.0,11.15,11.2,11.0
1,06:30,07:00,5.0,11.05,11.2,11.0
2,07:00,08:00,59.0,11.07,11.0,11.3
3,07:30,08:00,59.0,11.23,11.0,11.3
4,08:00,09:00,185.0,11.45,11.3,11.9
5,08:30,09:00,185.0,11.75,11.3,11.9


**What it shows.** **Observed:** the 06:00 and 06:30 slots take the radiation stamped 07:00 (5 W/m²), the 07:00 and 07:30 slots the one stamped 08:00 (59), and so on; temperature is interpolated to each slot's middle (06:15 → 11.15 °C, between 11.2 at 06:00 and 11.0 at 07:00). The 10th of November is held twice (section 3.4), so the query keeps one capture per hour.

**Learned → design:** DWD's 30-minute weather: radiation from the stamp at the end of the slot's hour; instantaneous variables (temperature, cloud cover, wind) interpolated linearly to the slot's middle. The mistake this guards against, radiation shifted by an hour, would not fail any range check; it only makes the model worse in a way that looks like noise.

## 4. The benchmark: TEPCO's next-day forecast

**Step.** `AREA_ONCE_YOSOKU.csv` holds TEPCO's forecast for the next day, in kWh per slot (÷ 500 for MW), wind and solar together, slot codes 1–48. It is captured at 19:00 and 22:00 JST. Each capture's values become versions as for the actuals, known at the fetch time: after `T`, so never an input, only something to compare with afterwards.

In [13]:
benchmark = sql("""
    WITH observed AS (
      SELECT
        TIMESTAMP_ADD(
          TIMESTAMP(DATETIME(PARSE_DATE('%Y%m%d', target_date), TIME(0, 0, 0)), 'Asia/Tokyo'),
          INTERVAL 30 * (slot_code - 1) MINUTE
        ) AS slot_start,
        wind_solar_kwh / 500 AS wind_solar_mw, fetched_at, source_time
      FROM tepco_area_forecast_next_day
    ),
    changes AS (
      SELECT *, LAG(wind_solar_mw) OVER (PARTITION BY slot_start ORDER BY fetched_at) AS previous,
             ROW_NUMBER() OVER (PARTITION BY slot_start ORDER BY fetched_at) AS sighting
      FROM observed
    )
    SELECT FORMAT_TIMESTAMP('%Y-%m-%d %H:%M:%S', fetched_at, 'Asia/Tokyo') AS captured_jst,
           FORMAT_TIMESTAMP('%H:%M:%S', MAX(source_time), 'Asia/Tokyo') AS file_written_jst,
           DATE(MIN(slot_start), 'Asia/Tokyo') AS target_day,
           COUNT(*) AS slots, COUNTIF(sighting = 1) AS first_versions,
           COUNTIF(sighting > 1 AND wind_solar_mw IS DISTINCT FROM previous) AS changed,
           ROUND(MAX(wind_solar_mw)) AS peak_wind_solar_mw
    FROM changes
    GROUP BY fetched_at
    ORDER BY fetched_at
""")
benchmark

[0.0 MB scanned]


,captured_jst,file_written_jst,target_day,slots,first_versions,changed,peak_wind_solar_mw
0,2026-11-15 19:00:18,18:50:12,2026-11-16,48,48,0,12295.0
1,2026-11-15 22:00:23,21:50:03,2026-11-16,48,0,0,12295.0
2,2026-11-16 19:00:47,18:50:16,2026-11-17,48,48,0,2784.0
3,2026-11-16 22:00:32,21:50:02,2026-11-17,48,0,0,2784.0
4,2026-11-16 22:00:53,21:50:02,2026-11-17,48,0,0,2784.0


**What it shows.** **Observed:** on both days so far, the 19:00 capture holds TEPCO's forecast for the next day, 48 slots (peaking at 12,295 MW of wind and solar for 16 November and 2,784 MW for the 17th; kWh per slot ÷ 500), and the 22:00 capture repeats it exactly. TEPCO writes the file twice, at about 18:50 and again at about 21:50, with the same values: only its update line changes. On 16 November the 22:00 capture ran twice, 21 seconds apart, into two identical files; the scheduler's log shows it started the job twice, both attempts answered. The 12:24 capture of 15 November, made before the file is filled, held only the header.

**Learned → design:** one version per slot, known at 19:00 on D−1, ten hours after `T`. The benchmark is compared with actual solar plus wind after the fact and never enters a feature. The 22:00 capture stays as a guard, since the file is cleared at about 23:13 and a late 19:00 publication would otherwise be lost. A repeated capture adds no version, so a job started twice costs a file, never a wrong value.

## 5. `as_of`: one value per slot, as known at `T`

### 5.1 Two shapes

DWS reads every input through `as_of(T)`: per key, the latest version with `known_at ≤ T`. Two ways to write it:

- **Per `T`**: filter `known_at ≤ @t`, then keep the newest version per key (`QUALIFY ROW_NUMBER() …`). Simple, but one query per decision time
- **As intervals**: give each version `known_until`, the next version's `known_at`, and join decision times on `known_at ≤ t < known_until`. One query serves every decision time in a backtest

**Step: build the lag feature both ways** (D−2's solar, slot by slot) and compare. The interval form runs for every target day of the training window in one pass.

In [14]:
INTERVALS = f"""
  SELECT *, LEAD(known_at) OVER (PARTITION BY slot_start, measure ORDER BY known_at) AS known_until
  FROM ({VERSIONS})
"""
DECISIONS = """
  SELECT d AS target_day,
         TIMESTAMP(DATETIME(DATE_SUB(d, INTERVAL 1 DAY), TIME(9, 0, 0)), 'Asia/Tokyo') AS t
  FROM UNNEST(GENERATE_DATE_ARRAY('2024-02-17', '2026-11-17')) AS d
"""

all_t = sql(f"""
    WITH v AS ({INTERVALS}), decisions AS ({DECISIONS})
    SELECT target_day, t, v.slot_start, v.value_mw, v.known_at, v.basis
    FROM decisions
    JOIN v ON v.measure = 'solar'
          AND DATE(v.slot_start, 'Asia/Tokyo') = DATE_SUB(target_day, INTERVAL 2 DAY)
          AND v.known_at <= t AND (v.known_until IS NULL OR t < v.known_until)
""")
per_t = all_t.groupby("target_day")["slot_start"].agg(["count", "nunique"])
print(
    "decision times:",
    len(per_t),
    "| slots per decision, min–max:",
    per_t["count"].min(),
    "–",
    per_t["count"].max(),
    "| any slot twice:",
    bool((per_t["count"] != per_t["nunique"]).any()),
)
print("basis of the rows used:", all_t["basis"].value_counts().to_dict())

[7.0 MB scanned]
decision times: 1005 | slots per decision, min–max: 48 – 48 | any slot twice: False
basis of the rows used: {'rule': 48212, 'captured': 28}


**Step: the per-`T` form for three decision times**, checked row for row against the interval form.

In [15]:
checks = []
for day in ("2024-02-17", "2025-08-01", "2026-11-17"):
    target = pd.Timestamp(day).date()
    t = pd.Timestamp(f"{target - pd.Timedelta(days=1)} 09:00", tz="Asia/Tokyo")
    one = (
        sql(
            f"""
        SELECT slot_start, value_mw, known_at
        FROM ({VERSIONS})
        WHERE measure = 'solar' AND known_at <= @t
          AND DATE(slot_start, 'Asia/Tokyo') = DATE_SUB(DATE(@t, 'Asia/Tokyo'), INTERVAL 1 DAY)
        QUALIFY ROW_NUMBER() OVER (PARTITION BY slot_start ORDER BY known_at DESC) = 1
    """,
            t=t,
        )
        .sort_values("slot_start")
        .reset_index(drop=True)
    )
    batch = (
        all_t[all_t["target_day"] == target][["slot_start", "value_mw", "known_at"]]
        .sort_values("slot_start")
        .reset_index(drop=True)
    )
    checks.append({"target_day": target, "rows": len(one), "identical": one.equals(batch)})
pd.DataFrame(checks)

[7.0 MB scanned]


[7.0 MB scanned]


[7.0 MB scanned]


,target_day,rows,identical
0,2024-02-17,48,True
1,2025-08-01,48,True
2,2026-11-17,48,True


**What it shows.** **Observed:**

- The interval form serves **all 1,005 decision times of the training window in one query**, returning exactly 48 slots of D−2 for each and never a slot twice. The rows it used are 48,212 by rule and 28 captured: the newest decision times already reach the capture era
- The per-`T` form gives the same rows, value for value, on three decision times from the window's start, middle and end
- Both scan the same 6.7 MB: the cost is building the versions, not the number of decision times

**Learned → design:** DWD materialises each versioned model with `known_until` (the next version's `known_at`, null for the newest). The `as_of` macro is then a range join, `known_at ≤ t AND (known_until IS NULL OR t < known_until)`, and DWS builds features for every decision time in one pass, which is what makes a full-history backtest one query.

### 5.2 Future poison

**Step.** Add a fake version to one slot of D−2, with an absurd value and a `known_at` one minute after `T`. `as_of(T)` must not see it; `as_of(T + 2 minutes)` must. This is the step-6 test in miniature, on the real versions.

In [16]:
target = pd.Timestamp("2025-08-01").date()
t = pd.Timestamp(f"{target - pd.Timedelta(days=1)} 09:00", tz="Asia/Tokyo")
slot = pd.Timestamp(f"{target - pd.Timedelta(days=2)} 12:00", tz="Asia/Tokyo")

POISONED = f"""
  SELECT slot_start, measure, value_mw, known_at FROM ({VERSIONS}) WHERE measure = 'solar'
  UNION ALL
  SELECT @slot, 'solar', 999999, TIMESTAMP_ADD(@t, INTERVAL 1 MINUTE)
"""
for label, at in (("T", t), ("T + 2 min", t + pd.Timedelta(minutes=2))):
    seen = sql(
        f"""
        SELECT value_mw FROM ({POISONED})
        WHERE slot_start = @slot AND known_at <= @at
        QUALIFY ROW_NUMBER() OVER (PARTITION BY slot_start ORDER BY known_at DESC) = 1
    """,
        slot=slot,
        t=t,
        at=at,
    )
    print(f"as_of({label}) for the 12:00 slot of {slot.date()}: {seen['value_mw'].item()}")

[7.0 MB scanned]
as_of(T) for the 12:00 slot of 2025-07-30: 14731


[7.0 MB scanned]
as_of(T + 2 min) for the 12:00 slot of 2025-07-30: 999999


**What it shows.** **Observed:** at `T`, the 12:00 slot of 2025-07-30 reads 14,731 MW, its real value. A fake version with 999,999 MW and `known_at` one minute after `T` is invisible at `T` and visible two minutes later.

**Learned → design:** the step-6 future-poison test works on these versions as they are: poison rows are ordinary versions with a later `known_at`, and the range join excludes them without any special case.

### 5.3 What `T` can see of D−1

**Step.** Lags use D−2, not D−1, because D−1 is only partly known at 09:00 on D−1 (PRD). Count how many of D−1's slots `as_of(T)` returns, over the training window.

In [17]:
sql(f"""
    WITH v AS ({INTERVALS}), decisions AS ({DECISIONS})
    SELECT COUNT(*) AS decision_times, MIN(slots) AS min_slots, MAX(slots) AS max_slots,
           APPROX_TOP_COUNT(slots, 3) AS most_common
    FROM (
      SELECT target_day, COUNT(*) AS slots
      FROM decisions
      JOIN v ON v.measure = 'solar'
            AND DATE(v.slot_start, 'Asia/Tokyo') = DATE_SUB(target_day, INTERVAL 1 DAY)
            AND v.known_at <= t AND (v.known_until IS NULL OR t < v.known_until)
      GROUP BY target_day
    )
""")

[7.0 MB scanned]


,decision_times,min_slots,max_slots,most_common
0,1005,16,17,"[{'value': 16, 'count': 1004}, {'value': 17, 'count': 1}]"


**What it shows.** **Observed:** at `T`, `as_of` returns 16 of D−1's slots (00:00 to 07:30) for 1,004 of the 1,005 decision times, and 17 for the one in the capture era: the 08:45 capture already held the 08:00 slot.

**Learned → design:** the history rule (slot end + 1 hour) is one slot more cautious than the captures, as notebook 02 expected. D−2 is complete either way, so lags and persistence use D−2, as planned, and the two eras give the model the same inputs.

## 6. Solar capacity: `known_at` in SQL

[Notebook 03](03_solar_capacity.ipynb) worked out, in pandas and from files it downloaded itself, when each quarterly release of solar capacity became known: at the end of the latest announcement date any copy of the summary page showed, never before its files' `Last-Modified`, and a release never announced at `Last-Modified` + 16 days. This section writes that rule over ODS, the way `dwd_capacity_releases` will, with notebook 03's printed tables as the answer key, and asks what changes now that ODS holds the evidence and the page is captured every day (since 08:45 on 2026-11-17).

| Question | Where |
| --- | --- |
| Does ODS hold the evidence the rule used, and what does each column mean? | 6.1 |
| Does one query reproduce notebook 03's dates for every release? | 6.2 |
| Can a later copy of the page move an old release's `known_at`? | 6.3 |
| How long does each date rest on the publisher's word alone? | 6.4 |
| Which releases can the capacity trend reach in the training window? | 6.5 |
| What happens at capture start, and to a release replaced later? | 6.6 |
| Does the TEPCO area's figure match, and what does each part cost? | 6.7 |

### 6.1 The evidence in ODS

**Step.** Every stored copy of the summary page, split by who took it (the Wayback Machine, or our capture), with when it was taken and when we fetched it. `source_time` is the time a file carries, else its `Last-Modified`; for an archived copy the parser sets it to the archive's own time (`Memento-Datetime`).

In [18]:
sql("""
    WITH copies AS (
      SELECT f.path, f.load_type, f.status, f.fetched_at,
             REGEXP_CONTAINS(f.path, r'_wayback_') AS archived,
             MIN(n.source_time) AS source_time, COUNT(n.ref_date) AS announcements
      FROM raw_files AS f
      LEFT JOIN meti_fit_notices AS n ON n.source_file = f.path
      WHERE f.ods_table = 'meti_fit_notices'
      GROUP BY f.path, f.load_type, f.status, f.fetched_at
    )
    SELECT IF(archived, 'archived', 'ours') AS taken_by, load_type, COUNT(*) AS copies,
           COUNTIF(status = 'empty') AS with_no_announcement,
           COUNTIF(source_time IS NULL) AS no_source_time,
           FORMAT_TIMESTAMP('%Y-%m-%d', MIN(source_time), 'Asia/Tokyo') AS first_taken,
           FORMAT_TIMESTAMP('%Y-%m-%d', MAX(source_time), 'Asia/Tokyo') AS last_taken,
           FORMAT_TIMESTAMP('%Y-%m-%d %H:%M', MIN(fetched_at), 'Asia/Tokyo') AS first_fetched,
           FORMAT_TIMESTAMP('%Y-%m-%d %H:%M', MAX(fetched_at), 'Asia/Tokyo') AS last_fetched
    FROM copies
    GROUP BY taken_by, load_type
    ORDER BY taken_by, load_type
""")

[0.3 MB scanned]


,taken_by,load_type,copies,with_no_announcement,no_source_time,first_taken,last_taken,first_fetched,last_fetched
0,archived,initial,52,1,1,2019-07-30,2026-10-27,2026-11-16 09:40,2026-11-16 09:42
1,ours,daily,1,0,1,NaN,NaN,2026-11-17 08:45,2026-11-17 08:45
2,ours,initial,1,0,1,NaN,NaN,2026-11-16 09:40,2026-11-16 09:40


**Step: the releases ODS holds files for, against the quarters some copy announces.** A release's files are named by the month they describe (`A_pref202603.xlsx`); the manifest lists them.

In [19]:
sql("""
    WITH held AS (
      SELECT DISTINCT
        LAST_DAY(PARSE_DATE('%Y%m', REGEXP_EXTRACT(path, r'_[AB]_(?:pref|city)(\\d{6})\\.xlsx$')))
          AS ref_date
      FROM raw_files
      WHERE ods_table IN ('meti_fit_capacity_pref', 'meti_fit_capacity_city')
    ),
    announced AS (SELECT DISTINCT ref_date FROM meti_fit_notices)
    SELECT
      (SELECT COUNT(*) FROM held) AS releases_held,
      (SELECT COUNT(*) FROM announced) AS quarters_announced,
      ARRAY(SELECT CAST(ref_date AS STRING) FROM held
            WHERE ref_date NOT IN (SELECT ref_date FROM announced)) AS held_never_announced,
      ARRAY(SELECT CAST(ref_date AS STRING) FROM announced
            WHERE ref_date NOT IN (SELECT ref_date FROM held)) AS announced_not_held
""")

[0.2 MB scanned]


,releases_held,quarters_announced,held_never_announced,announced_not_held
0,33,33,[2017-06-30],[2018-03-31]


**What it shows.** **Observed:**

- 54 copies of the page: 52 taken by the archive, from October 2018 to 27 October 2026 (the one from October 2018 announces nothing, so the dated ones start on 30 July 2019), and two of ours, the live page fetched with the initial load at 09:40 on 16 November and the first daily copy at 08:45 on the 17th
- Every archived copy that announces anything carries the archive's time in `source_time`; **ours carry none**, since a live page states no time of its own. Every copy's `fetched_at` is our fetch: 16 November for all 52 archived ones
- ODS holds 33 releases and the copies announce 33 quarters, but not the same 33: **end of June 2017 is held and never announced** (its `.xlsx` was first linked in 2024, section 6.6), and end of March 2018 is announced but not held (an `.xls`, left out)

**Learned → design:** when a copy was taken is `source_time` for an archived copy and `fetched_at` for ours, `COALESCE(source_time, fetched_at)`; `fetched_at` alone would date all 52 archived copies to 16 November 2026. Everything the rule needs is in ODS, and each file's `Last-Modified` is in the manifest, so the dates never read the large tables.

### 6.2 The rule in one query, against notebook 03

**Step.** The rule as one query, the first draft of `dwd_capacity_releases`. Its inputs come from the notices and the manifest only: the latest date any copy taken before capture start showed, and each file's `Last-Modified` from its HTTP headers. A release counts once both its tables are held. `rule` says which part of the rule set the date; `basis` is `rule` when that date falls before the source's capture start, as for every source (section 1), and `captured` otherwise, when the date is the fetch of the capture that first held both files.

In [20]:
RELEASES = """
    WITH capture_start AS (
      SELECT MIN(fetched_at) AS t FROM raw_files
      WHERE source = 'meti_fit_capacity' AND load_type = 'daily'
    ),
    -- each announcement as one copy of the page showed it, and when that copy was taken: the
    -- archive's time for an archived copy, our fetch for our own (whose source_time is empty)
    shown AS (
      SELECT ref_date, announced, COALESCE(source_time, fetched_at) AS taken_at
      FROM meti_fit_notices
    ),
    -- the rule reads only copies taken before capture start, so later copies can't move it
    announced AS (
      SELECT ref_date, MAX(announced) AS announced
      FROM shown
      WHERE taken_at < (SELECT t FROM capture_start)
      GROUP BY ref_date
    ),
    files AS (
      SELECT
        LAST_DAY(PARSE_DATE('%Y%m', REGEXP_EXTRACT(path, r'_[AB]_(?:pref|city)(\\d{6})\\.xlsx$')))
          AS ref_date,
        ods_table, fetched_at,
        PARSE_TIMESTAMP('%a, %d %b %Y %H:%M:%S %z', last_modified) AS last_modified
      FROM raw_files
      WHERE ods_table IN ('meti_fit_capacity_pref', 'meti_fit_capacity_city') AND status = 'parsed'
    ),
    held AS (
      SELECT ref_date, COUNT(*) AS tables,
             MAX(IF(ods_table = 'meti_fit_capacity_pref', last_modified, NULL)) AS a_last_modified,
             MAX(IF(ods_table = 'meti_fit_capacity_city', last_modified, NULL)) AS b_last_modified,
             MAX(last_modified) AS last_modified,
             MAX(first_fetched) AS both_fetched
      FROM (
        SELECT ref_date, ods_table, MAX(last_modified) AS last_modified,
               MIN(fetched_at) AS first_fetched
        FROM files
        GROUP BY ref_date, ods_table
      )
      GROUP BY ref_date
    ),
    ruled AS (
      SELECT held.*, announced.announced,
        IF(announced.announced IS NOT NULL,
           GREATEST(TIMESTAMP(DATE_ADD(announced.announced, INTERVAL 1 DAY), 'Asia/Tokyo'),
                    held.last_modified),
           TIMESTAMP(DATE_ADD(DATE(held.last_modified, 'Asia/Tokyo'), INTERVAL 17 DAY), 'Asia/Tokyo'))
          AS rule_time
      FROM held LEFT JOIN announced USING (ref_date)
    )
    SELECT ref_date, announced, a_last_modified, b_last_modified, tables,
      CASE WHEN announced IS NULL THEN 'margin'
           WHEN rule_time = TIMESTAMP(DATE_ADD(announced, INTERVAL 1 DAY), 'Asia/Tokyo')
             THEN 'announced'
           ELSE 'last_modified' END AS rule,
      IF(rule_time < (SELECT t FROM capture_start), 'rule', 'captured') AS basis,
      IF(rule_time < (SELECT t FROM capture_start), rule_time, both_fetched) AS known_at
    FROM ruled
"""
releases = sql(RELEASES + " ORDER BY ref_date")
print(releases.groupby(["basis", "rule"]).size().to_dict(), "| tables per release:",
      releases["tables"].value_counts().to_dict())
releases.assign(known_at=releases["known_at"].dt.tz_convert("Asia/Tokyo").dt.strftime("%Y-%m-%d %H:%M"))[
    ["ref_date", "announced", "rule", "basis", "known_at"]
]

[0.3 MB scanned]
{('rule', 'announced'): 23, ('rule', 'last_modified'): 9, ('rule', 'margin'): 1} | tables per release: {np.int64(2): 33}


,ref_date,announced,rule,basis,known_at
0,2017-06-30,NaT,margin,rule,2024-02-23 00:00
1,2018-06-30,2018-12-12,announced,rule,2018-12-13 00:00
2,2018-09-30,2019-02-05,last_modified,rule,2019-04-11 10:17
3,2018-12-31,2019-04-11,announced,rule,2019-04-12 00:00
4,2019-03-31,2019-07-31,announced,rule,2019-08-01 00:00
5,2019-06-30,2019-10-29,last_modified,rule,2021-10-15 12:15
6,2019-09-30,2020-01-27,last_modified,rule,2021-10-15 12:15
7,2019-12-31,2020-04-28,last_modified,rule,2021-10-15 12:15
8,2020-03-31,2020-07-10,last_modified,rule,2021-10-15 12:16
9,2020-06-30,2020-10-29,last_modified,rule,2021-10-15 12:16


**Step: compare with notebook 03.** Its section 2 printed, for every release, the latest announcement date, table A's `Last-Modified` and how many days later table B's went up, and the `known_at` of the twelve newest. Those printed values are the answer key, copied here for the 33 releases ODS holds.

In [21]:
NOTEBOOK_03_INPUTS = pd.DataFrame(
    [
        ("2017-06-30", None, "2024-02-06 16:29", 0.0),
        ("2018-06-30", "2018-12-12", "2018-12-12 12:10", 0.0),
        ("2018-09-30", "2019-02-05", "2019-04-11 10:17", 0.0),
        ("2018-12-31", "2019-04-11", "2019-04-11 10:17", 0.0),
        ("2019-03-31", "2019-07-31", "2019-07-31 08:47", 0.0),
        ("2019-06-30", "2019-10-29", "2019-10-25 14:47", 720.9),
        ("2019-09-30", "2020-01-27", "2020-01-21 10:23", 633.1),
        ("2019-12-31", "2020-04-28", "2020-04-28 14:44", 534.9),
        ("2020-03-31", "2020-07-10", "2020-07-10 15:27", 461.9),
        ("2020-06-30", "2020-10-29", "2020-10-27 17:35", 352.8),
        ("2020-09-30", "2021-01-29", "2021-01-29 15:01", 262.1),
        ("2020-12-31", "2021-05-14", "2021-06-15 17:19", 125.0),
        ("2021-03-31", "2021-07-21", "2021-07-21 17:21", 85.1),
        ("2021-06-30", "2021-10-29", "2021-10-29 17:16", 0.0),
        ("2021-09-30", "2022-01-27", "2022-01-20 09:19", 0.0),
        ("2021-12-31", "2022-05-13", "2022-05-09 09:00", 0.0),
        ("2022-03-31", "2022-07-29", "2022-07-26 13:47", 0.0),
        ("2022-06-30", "2022-11-09", "2022-10-24 14:48", 0.0),
        ("2022-09-30", "2023-02-17", "2023-02-10 19:23", 0.0),
        ("2022-12-31", "2023-05-09", "2023-05-01 14:58", 0.0),
        ("2023-03-31", "2023-08-10", "2023-08-09 00:00", 0.0),
        ("2023-06-30", "2023-11-16", "2023-11-16 10:03", 0.0),
        ("2023-09-30", "2024-02-06", "2024-02-06 16:24", 0.0),
        ("2023-12-31", "2024-04-30", "2024-04-30 15:19", 0.0),
        ("2024-03-31", "2024-08-07", "2024-08-07 15:28", 0.0),
        ("2024-06-30", "2024-12-11", "2024-12-11 17:35", 0.0),
        ("2024-09-30", "2025-03-13", "2025-03-11 14:25", 0.0),
        ("2024-12-31", "2025-05-09", "2025-05-09 12:02", 0.0),
        ("2025-03-31", "2025-08-20", "2025-08-19 21:51", 0.0),
        ("2025-06-30", "2025-12-09", "2025-12-09 13:37", 0.0),
        ("2025-09-30", "2026-03-19", "2026-03-13 12:00", 0.0),
        ("2025-12-31", "2026-06-04", "2026-06-04 09:02", 0.0),
        ("2026-03-31", "2026-08-18", "2026-08-18 13:42", 0.0),
    ],
    columns=["ref_date", "announced", "a_last_modified_jst", "b_minus_a_days"],
)
NOTEBOOK_03_KNOWN_AT = {  # the twelve it printed, end of the announcement day, JST
    "2023-06-30": "2023-11-17 00:00", "2023-09-30": "2024-02-07 00:00",
    "2023-12-31": "2024-05-01 00:00", "2024-03-31": "2024-08-08 00:00",
    "2024-06-30": "2024-12-12 00:00", "2024-09-30": "2025-03-14 00:00",
    "2024-12-31": "2025-05-10 00:00", "2025-03-31": "2025-08-21 00:00",
    "2025-06-30": "2025-12-10 00:00", "2025-09-30": "2026-03-20 00:00",
    "2025-12-31": "2026-06-05 00:00", "2026-03-31": "2026-08-19 00:00",
}
key = NOTEBOOK_03_INPUTS.assign(
    ref_date=lambda d: pd.to_datetime(d["ref_date"]).dt.date,
    announced=lambda d: pd.to_datetime(d["announced"]).dt.date,
)
ours = releases.assign(
    a_last_modified_jst=releases["a_last_modified"].dt.tz_convert("Asia/Tokyo").dt.strftime("%Y-%m-%d %H:%M"),
    b_minus_a_days=((releases["b_last_modified"] - releases["a_last_modified"]).dt.total_seconds() / 86400).round(1),
    known_at_jst=releases["known_at"].dt.tz_convert("Asia/Tokyo").dt.strftime("%Y-%m-%d %H:%M"),
)
both = ours.merge(key, on="ref_date", suffixes=("", "_03"))
same_announced = (both["announced"] == both["announced_03"]) | (both["announced"].isna() & both["announced_03"].isna())
print("releases compared:", len(both), "of", len(ours))
print("same latest announcement date:", int(same_announced.sum()))
print("same table A Last-Modified, to the minute:", int((both["a_last_modified_jst"] == both["a_last_modified_jst_03"]).sum()))
print("same table B lag, to 0.1 day:", int(((both["b_minus_a_days"] - both["b_minus_a_days_03"]).abs() <= 0.1).sum()))
printed = both[both["ref_date"].astype(str).isin(NOTEBOOK_03_KNOWN_AT)]
print("same known_at where notebook 03 printed it:",
      int((printed["known_at_jst"] == printed["ref_date"].astype(str).map(NOTEBOOK_03_KNOWN_AT)).sum()), "of", len(printed))

releases compared: 33 of 33
same latest announcement date: 33
same table A Last-Modified, to the minute: 33
same table B lag, to 0.1 day: 33
same known_at where notebook 03 printed it: 12 of 12


**What it shows.** **Observed:** one query of 0.3 MB reproduces notebook 03:

- All 33 releases have notebook 03's inputs: the same latest announcement date, table A's `Last-Modified` to the minute and table B's lag to a tenth of a day; and the twelve `known_at` it printed match
- 23 releases are dated by their announcement, 9 by a later `Last-Modified` (end of September 2018, and the eight from end of June 2019 to end of March 2021, whose table B was re-uploaded in October 2021) and one by the margin: **end of June 2017, known on 23 February 2024**
- Every release is `basis = rule`: all 33 rule dates fall before capture start (section 6.6)

**Learned → design:** `dwd_capacity_releases` is this query, plain SQL over the notices and the manifest. Notebook 03's 13 releases not dated by announcement are these 10 plus the three `.xls` releases ODS doesn't hold.

### 6.3 Later copies of the page

**Step.** Notebook 03 found the publisher editing its own dates. Taking the latest date any copy showed means a later copy can only move a date later, never earlier; but it can still move it, after the fact. Replay the archive in the order its copies were taken: for each release, every copy that raised the latest date shown so far, and how long after the release was first listed. Then the copies that showed an *earlier* date than an earlier copy had (the backward edits), which the rule ignores.

In [22]:
sql("""
    WITH per_copy AS (
      SELECT ref_date, source_file, COALESCE(source_time, fetched_at) AS taken_at,
             MAX(announced) AS shown
      FROM meti_fit_notices
      GROUP BY ref_date, source_file, taken_at
    ),
    running AS (
      SELECT *,
        MAX(shown) OVER (PARTITION BY ref_date ORDER BY taken_at
                         ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS latest_before,
        MIN(taken_at) OVER (PARTITION BY ref_date) AS listed_from
      FROM per_copy
    )
    SELECT ref_date, IF(shown > latest_before, 'later', 'earlier') AS edit,
           latest_before, shown,
           FORMAT_TIMESTAMP('%Y-%m-%d', listed_from, 'Asia/Tokyo') AS first_listed,
           FORMAT_TIMESTAMP('%Y-%m-%d', MIN(taken_at), 'Asia/Tokyo') AS first_copy_showing_it,
           TIMESTAMP_DIFF(MIN(taken_at), listed_from, DAY) AS days_after_first_listed
    FROM running
    WHERE shown != latest_before
    GROUP BY ref_date, edit, latest_before, shown, listed_from
    ORDER BY ref_date, first_copy_showing_it
""")

[0.0 MB scanned]


,ref_date,edit,latest_before,shown,first_listed,first_copy_showing_it,days_after_first_listed
0,2019-03-31,later,2019-07-18,2019-07-31,2019-07-30,2019-08-19,19
1,2019-03-31,earlier,2019-07-31,2019-07-18,2019-07-30,2019-11-05,98
2,2021-12-31,earlier,2022-05-13,2022-05-09,2022-05-25,2022-10-24,152
3,2022-03-31,earlier,2022-07-29,2022-07-26,2022-10-24,2022-12-05,41
4,2022-06-30,earlier,2022-11-09,2022-10-24,2022-12-05,2023-03-28,113
5,2023-03-31,later,2023-08-09,2023-08-10,2023-10-03,2024-03-03,151
6,2024-09-30,earlier,2025-03-13,2025-03-11,2025-04-15,2025-05-19,33


**What it shows.** **Observed:** in the order the archive took its copies,

- **Two releases had their date moved later by a later copy:** end of March 2019 by 13 days, 19 days after it was first listed (the heading changed to "updated 31 July": a re-publication), and **end of March 2023 by one day, 151 days after it was first listed**
- Five had their date moved earlier, up to 152 days after first listing. The latest date wins, so those never move `known_at`

**Learned → design:** under "the latest date any copy showed", a copy can move a release's `known_at` months after the fact. With a copy stored every day, the newest releases' dates would stay open to change, and a replay of a decision time between two such edits would change with them: the backfill-stability test would fail. So the history rule reads only copies taken before the source's capture start, which fixes the past. A release first seen after capture start is dated by our capture (section 6.6), and the page's dates don't matter for it.

### 6.4 The publisher's word

**Step.** The rule dates a release from the date the page states, which is earlier than any copy we hold that shows it, since the archive took copies weeks apart. For each release: how many days its `known_at` comes before the first copy listing it, by which part of the rule set it, and separately for the releases the trend can reach in the training window (end of 2022 on, section 6.5).

In [23]:
sql(f"""
    WITH releases AS ({RELEASES}),
    listing AS (
      SELECT ref_date, MIN(COALESCE(source_time, fetched_at)) AS first_listed
      FROM meti_fit_notices
      GROUP BY ref_date
    )
    SELECT rule, ref_date >= '2022-12-31' AS trend_reaches_it, COUNT(*) AS releases,
           APPROX_QUANTILES(TIMESTAMP_DIFF(first_listed, known_at, DAY), 2)[OFFSET(1)]
             AS median_days_before_first_listed,
           MAX(TIMESTAMP_DIFF(first_listed, known_at, DAY)) AS most_days_before_first_listed
    FROM releases JOIN listing USING (ref_date)
    GROUP BY rule, trend_reaches_it
    ORDER BY rule, trend_reaches_it
""")

[0.3 MB scanned]


,rule,trend_reaches_it,releases,median_days_before_first_listed,most_days_before_first_listed
0,announced,False,9,25,229
1,announced,True,14,25,107
2,last_modified,False,9,-327,110


**What it shows.** **Observed:**

- For the 14 releases the trend can reach (end of 2022 on), `known_at` comes a median of 25 days before the first copy listing the release, at most 107 (end of June 2023: known on 17 November 2023, first in a copy of 3 March 2024). The nine older releases dated by announcement look the same, at most 229 days
- Where `Last-Modified` set the date, it comes long after the first listing (median 327 days): the floor is far more cautious than the page

**Learned → design:** the archive took copies weeks or months apart, so for weeks the page's own date is the only evidence of the day itself. It holds up: notebook 03 found the files going up on the announcement day or a few days before (section 2), and the floor never lets a release be known before its files. The rule stays as it is; from capture start, our daily copy replaces the archive as the evidence.

### 6.5 Which releases the trend reaches

**Step.** DWS extends the newest figure to D by a straight line through the newest four releases known at `T` (notebook 03, section 7). "Newest" can mean the quarter a release describes, or when it became known; notebook 03 sorted by the quarter. For every decision time of the training window, pick the four both ways and see which releases each uses.

In [24]:
sql(f"""
    WITH releases AS ({RELEASES}),
    decisions AS (
      SELECT TIMESTAMP(DATETIME(DATE_SUB(d, INTERVAL 1 DAY), TIME(9, 0, 0)), 'Asia/Tokyo') AS t
      FROM UNNEST(GENERATE_DATE_ARRAY('2024-02-17', '2026-11-18')) AS d
    ),
    known AS (SELECT decisions.t, releases.* FROM decisions JOIN releases ON releases.known_at <= decisions.t),
    by_quarter AS (
      SELECT * FROM known QUALIFY ROW_NUMBER() OVER (PARTITION BY t ORDER BY ref_date DESC) <= 4
    ),
    by_known_at AS (
      SELECT * FROM known QUALIFY ROW_NUMBER() OVER (PARTITION BY t ORDER BY known_at DESC) <= 4
    )
    SELECT 'newest by the quarter described' AS newest, COUNT(DISTINCT t) AS decision_times,
           MIN(ref_date) AS oldest_release_used,
           COUNTIF(rule != 'announced') AS uses_of_a_release_not_dated_by_announcement,
           COUNT(DISTINCT IF(ref_date = '2017-06-30', t, NULL)) AS decision_times_using_2017_06
    FROM by_quarter
    UNION ALL
    SELECT 'newest by known_at', COUNT(DISTINCT t), MIN(ref_date), COUNTIF(rule != 'announced'),
           COUNT(DISTINCT IF(ref_date = '2017-06-30', t, NULL))
    FROM by_known_at
""")

[0.3 MB scanned]


,newest,decision_times,oldest_release_used,uses_of_a_release_not_dated_by_announcement,decision_times_using_2017_06
0,newest by known_at,1006,2017-06-30,385,385
1,newest by the quarter described,1006,2022-12-31,0,0


**What it shows.** **Observed:** over the 1,006 decision times of the training window,

- Newest **by the quarter described**, the trend never uses a release older than end of 2022, nor any release dated by `Last-Modified` or the margin
- Newest **by `known_at`**, it uses end of June 2017 on 385 decision times: a release known only from 23 February 2024 counts as one of the newest for more than a year

**Learned → design:** the capacity estimate takes the newest four releases by `ref_date` among those known at `T`, as notebook 03 did. A unit test pins it with a release known late, as end of June 2017 is.

### 6.6 Capture start, and a release replaced later

**Step.** The source's capture start, and the `basis` of every release against it. Then what the daily capture can see from now on: it stores the page every day and fetches each release's files **once, by name** (`FetchItem(once=True)` in `jtf/src/jtf/ingest/sources.py`), so it would miss a file the publisher replaces in place. Whether such a replacement shows on the page is answered by the links themselves: compare the file ids the page links for releases known to have been replaced (table B for 2019 to 2021, re-uploaded in October 2021 by their `Last-Modified`; end of June 2017, first linked in 2024), in copies taken before and after. This reads raw pages from the bucket, since ODS keeps the notices, not the links.

In [25]:
sql("""
    SELECT
      FORMAT_TIMESTAMP('%Y-%m-%d %H:%M', MIN(IF(load_type = 'daily', fetched_at, NULL)), 'Asia/Tokyo')
        AS capture_start_jst,
      COUNTIF(load_type = 'daily') AS daily_files,
      COUNTIF(load_type = 'daily' AND ods_table != 'meti_fit_notices') AS daily_release_files
    FROM raw_files
    WHERE source = 'meti_fit_capacity'
""")

[0.1 MB scanned]


,capture_start_jst,daily_files,daily_release_files
0,2026-11-17 08:45,1,0


In [26]:
from jtf.ingest.sources import fit_links
from jtf.storage import open_store

raw_url = os.environ.get("JTF_RAW_URL") or dict(
    line.split("=", 1) for line in Path("../.env").read_text().splitlines()
    if "=" in line and not line.startswith("#")
)["JTF_RAW_URL"]
store = open_store(raw_url)
pages = sql("""
    SELECT f.path,
           FORMAT_TIMESTAMP('%Y-%m-%d', COALESCE(MIN(n.source_time), MIN(f.fetched_at)), 'Asia/Tokyo')
             AS taken
    FROM raw_files AS f LEFT JOIN meti_fit_notices AS n ON n.source_file = f.path
    WHERE f.ods_table = 'meti_fit_notices'
      AND REGEXP_CONTAINS(f.path, r'wayback_(20210924|20211105|20231003|20240302)|^meti_fit_capacity/2026-11-17/')
    GROUP BY f.path ORDER BY taken
""")
links = {
    row.taken: {item.filename: item.params["file"] for item in fit_links(store.get(row.path))}
    for row in pages.itertuples()
}
watch = ["B_city201906.xlsx", "B_city202103.xlsx", "A_pref202103.xlsx", "A_pref201706.xlsx", "B_city201706.xlsx"]
pd.DataFrame({taken: [ids.get(name, "–") for name in watch] for taken, ids in links.items()}, index=watch)

[0.2 MB scanned]


,2021-09-24,2021-11-06,2023-10-03,2024-03-03,2026-11-17
B_city201906.xlsx,0150K000009VpeF,0150K000009VpeF,0150K000009VpeF,0150K000009VpeF,0150K000009VpeF
B_city202103.xlsx,0150K00000AnspB,0150K00000AnspB,0150K00000AnspB,0150K00000AnspB,0150K00000AnspB
A_pref202103.xlsx,0150K00000Ansox,0150K00000Ansox,0150K00000Ansox,0150K00000Ansox,0150K00000Ansox
A_pref201706.xlsx,–,–,–,015IT00000AyrbM,015IT00000AyrbM
B_city201706.xlsx,–,–,–,015IT00000AyrbR,015IT00000AyrbR


**What it shows.** **Observed:**

- Capture start is 08:45 on 2026-11-17, this source's first daily capture, which stored the page and no release file. All 33 rule dates fall before it, so every release is `basis = rule`
- **A replaced file keeps its link.** Table B for end of June 2019 and end of March 2021 was re-uploaded in October 2021, and its file id is the same in copies from September 2021, November 2021, October 2023, March 2024 and today. End of June 2017 is the other kind of change: new ids, first linked between October 2023 and March 2024

**Learned → design:** a release first seen after capture start is known at the fetch of the capture that first holds both its files, `basis = captured`, as for every source; end of June 2026, due between now and mid-December, will be the first. A release the publisher re-uploads in place is invisible to the capture as written: the page shows the same link, and a file already held is never fetched again. Seeing it needs each linked file's `Last-Modified` checked daily (a header request per file) and a re-fetch when it changes; the re-fetched file is then a new version of the release's values, known at that fetch.

### 6.7 The TEPCO area, and what each part costs

**Step.** The figure the features use: installed solar (new and transferred, under and over 10 kW) in the eight prefectures from table A and the 20 Shizuoka municipalities east of the Fuji River from table B, per release, each file as last parsed (notebook 03, section 5). Compared with what notebook 03 printed for the twelve newest releases, with the bytes this scans next to the release dates' (section 6.2).

In [27]:
TEPCO_PREFECTURES = ["茨城県", "栃木県", "群馬県", "埼玉県", "千葉県", "東京都", "神奈川県", "山梨県"]
SHIZUOKA_EAST = ["静岡県" + m for m in (
    "沼津市", "熱海市", "三島市", "富士宮市", "伊東市", "富士市", "御殿場市", "下田市", "裾野市", "伊豆市",
    "伊豆の国市", "賀茂郡東伊豆町", "賀茂郡河津町", "賀茂郡南伊豆町", "賀茂郡松崎町", "賀茂郡西伊豆町",
    "田方郡函南町", "駿東郡清水町", "駿東郡長泉町", "駿東郡小山町",
)]
quoted = lambda names: ", ".join(f"'{n}'" for n in names)
area = sql(f"""
    WITH solar_a AS (
      SELECT ref_date, area, value FROM meti_fit_capacity_pref
      WHERE sheet IN ('表A②－１', '表A②－２')
        AND path IN ('太陽光発電設備/10kW未満', '太陽光発電設備/10kW以上')
      QUALIFY ingested_at = MAX(ingested_at) OVER (PARTITION BY source_file)
    ),
    solar_b AS (
      SELECT ref_date, area, value FROM meti_fit_capacity_city
      WHERE sheet IN ('表B②－１', '表B②－２')
        AND path IN ('太陽光発電設備/10kW未満', '太陽光発電設備/10kW以上')
      QUALIFY ingested_at = MAX(ingested_at) OVER (PARTITION BY source_file)
    )
    SELECT ref_date, SUM(value) AS tepco_kw
    FROM (
      SELECT * FROM solar_a WHERE area IN ({quoted(TEPCO_PREFECTURES)})
      UNION ALL
      SELECT * FROM solar_b WHERE area IN ({quoted(SHIZUOKA_EAST)})
    )
    GROUP BY ref_date
    ORDER BY ref_date
""")
NOTEBOOK_03_GW = {  # section 5, the twelve it printed
    "2023-06-30": 18.56, "2023-09-30": 18.75, "2023-12-31": 18.97, "2024-03-31": 19.15,
    "2024-06-30": 19.31, "2024-09-30": 19.48, "2024-12-31": 19.69, "2025-03-31": 19.93,
    "2025-06-30": 20.03, "2025-09-30": 20.24, "2025-12-31": 20.41, "2026-03-31": 20.58,
}
area["tepco_mw"] = (area["tepco_kw"] / 1000).round()
area["notebook_03_gw"] = area["ref_date"].astype(str).map(NOTEBOOK_03_GW)
printed = area.dropna(subset=["notebook_03_gw"])
print("releases:", len(area), "| same as notebook 03 to 0.01 GW:",
      int(((printed["tepco_kw"] / 1e6).round(2) == printed["notebook_03_gw"]).sum()), "of", len(printed))
area[["ref_date", "tepco_mw", "notebook_03_gw"]].tail(5)

[2085.1 MB scanned]
releases: 33 | same as notebook 03 to 0.01 GW: 12 of 12


,ref_date,tepco_mw,notebook_03_gw
28,2025-03-31,19929.0,19.93
29,2025-06-30,20032.0,20.03
30,2025-09-30,20244.0,20.24
31,2025-12-31,20406.0,20.41
32,2026-03-31,20577.0,20.58


**What it shows.** **Observed:** the TEPCO area's capacity for all 33 releases, matching the twelve notebook 03 printed to 0.01 GW (20,577 MW at the end of March 2026). It scans **2,085 MB**, against 0.3 MB for the release dates: nearly all of the municipality table, since its layout (partitioned by `extraction_date`, clustered by file) can't narrow a filter on area.

**Learned → design:** two models, split by cost and by how often they change:

- `dwd_capacity_releases`: one row per release with `known_at` and `basis` (section 6.2), from the notices and the manifest. Cheap, rebuilt whenever the page is captured
- `dwd_capacity_area`: solar kW per release and area, the eight prefectures and Shizuoka east of the Fuji River kept apart for the weather-point weights, from tables A and B. The page arrives daily, so a full rebuild would scan 2 GB a day; the model reads only the ODS partitions parsed since its last run, so a day without a new release costs nothing. This is where ODS's partitions by fetch day pay off

DWS reads both through `as_of` on the release's `known_at`.

## The DWD models this leads to

| Model | One row per | Columns besides the key | From |
| --- | --- | --- | --- |
| `dwd_tepco_actuals` | Slot (JST start), measure, version | `value_mw`, `known_at`, `known_until`, `basis`, `first_seen`, `source_file`, `correction` | `ods.tepco_actuals`; capture start from `ods.raw_files` |
| `dwd_tepco_actuals_hourly` | Hour, measure | `value_mw` (万kWh × 10), `known_at` (rule) | `ods.tepco_actuals_yearly`; outside the training path |
| `dwd_weather_previous_runs` | Point, model, lead, hour | The six variables, `known_at` (rule), `basis = rule` | `ods.openmeteo_previous_runs` |
| `dwd_weather_single_runs` | Point, model, run, hour | The six variables, `known_at` (run + delay), `delay_source` (assumed or measured) | `ods.openmeteo_single_runs` |
| `dwd_weather_live` | Point, model, capture, hour | The six variables, `known_at` (fetch), `basis = captured` | `ods.openmeteo_live` |
| `dwd_weather_slots` | Point, model, source, slot | Radiation from the stamp at the end of the slot's hour; other variables interpolated to the slot's middle | The three weather models |
| `dwd_tepco_forecast_next_day` | Slot, version | `wind_solar_mw`, `known_at`, `known_until` | `ods.tepco_area_forecast_next_day` |
| `dwd_capacity_releases` | Release | `known_at`, `basis`, `rule` (announced, last_modified or margin), `announced`, both files' `Last-Modified` | `ods.meti_fit_notices`, `ods.raw_files` (section 6.2) |
| `dwd_capacity_area` | Release, area (eight prefectures, Shizuoka east of the Fuji River), version | Solar kW; `known_at`: the release's for its first version, the re-fetch's for a later one | `ods.meti_fit_capacity_pref`, `ods.meti_fit_capacity_city`, read incrementally by fetch day (section 6.7) |

The `as_of(model, t)` macro is the range join of section 5. Tests, beyond the unit tests already in the plan (a late backfill, capture start per source, repeats, corrections, re-parses, revised dates):

- Unique (key, `known_at`) in every versioned model; `known_until` after `known_at`; versions of a key contiguous
- `basis = rule` only before the source's capture start; `basis = captured` carries the `first_seen` of a file holding the value
- Every target day of the window has weather known at `T` (section 3.4) and 48 slots of D−2 (section 5.1)
- Capacity: a copy of the page taken after capture start never moves a rule-dated release; a release first held after capture start is `captured` at the fetch that holds both files; the trend's four releases are the newest by quarter, pinned with a release known late (sections 6.3, 6.5, 6.6)

## What this exploration made known

| We assumed | We found | So |
| --- | --- | --- |
| Versions and `known_at` need custom code | One window function and a `CASE` over all of `ods.tepco_actuals`: 146,787 versions in one 7 MB query (section 2) | They are plain dbt SQL; capture start is read from `ods.raw_files`, not a setting |
| Corrections would show up in daily captures | Three days of captures, 2,225 slots seen more than once, no value changed (section 2) | The versioning stays (it costs nothing); the monthly re-capture is what will find corrections |
| Data loaded after capture start could take the history rule | Three slots first seen by the 14:02 re-capture would have been dated 13:30 and 14:00 by the rule, before we had them (section 2.2) | The rule applies only before capture start, as planned; this real case becomes the late-backfill unit test |
| The yearly and monthly actuals need joining for training | They meet at 2024-01-31 and 2024-02-01, and the training window and its lags use only the monthly files (section 2.3) | `dwd_tepco_actuals` is the monthly files; the yearly ones get their own hourly model, off the training path |
| Lead 1 might be usable for some slots | At `T` it covers 6.9% of D's daylight, only up to 06:00 JST; lead 2 covers all of it (section 3.1) | Features use lead 2 only; leads 0 and 1 serve the F2 leak scenario |
| The newest Single Run known at `T` is a safe choice | With the assumed delays, each model's newest run becomes available exactly at `T` on 186 of 187 days; after missing runs it is days old, and on 9 November MSM's ends before D begins (section 3.2) | Until delays are measured, DWS uses the older run captured per model and day, and only a run that covers D's daylight |
| History and live capture meet at capture start | Target day 2026-11-15 had 9 of 24 hours of weather at `T`, and 2026-11-16 none. After the top-ups of 16 November every day has some, but lead 2 ends on the 15th, Single Runs on the 16th, and the live capture starts on the 17th (section 3.4) | The weather DWS picks a source per target day; a dbt test checks every target day has weather at `T` |
| A history load fetches an archive that doesn't change | Fetched twice, 1–14 November agrees on 26,959 of 27,063 rows; the 104 that differ are lead 0 of GSM and ECMWF for the hours just before the first fetch (section 3.4) | Weather is versioned like every source: a copy seen again adds nothing, a changed value is a second version known at the later fetch. Lead 2 never changed |
| Hourly weather maps onto slots as it is | Radiation is the mean of the hour before its stamp; temperature is instantaneous (section 3.5) | Both slots of an hour take the radiation stamped at its end; instantaneous variables are interpolated to slot middles |
| `as_of` runs once per decision time | A range join on `known_until` serves all 1,005 decision times in one pass, for the same 6.7 MB, and matches the per-`T` form row for row (section 5.1) | DWD materialises `known_until`; `as_of` is a range join; a backtest's features are one query |
| A future row could slip through a join | A poisoned version one minute after `T` is invisible at `T` and visible after (section 5.2) | The future-poison test needs no special case in the macro |
| `T` sees D−1 up to about 08:00 | 16 slots in the rule era, 17 in the capture era (section 5.3) | Lags use D−2, complete in both eras |
| The next-day forecast might change between 19:00 and 22:00 | On both days, the 22:00 capture repeated the 19:00 one; TEPCO rewrites the file at about 21:50 with the same values, and one 22:00 capture ran twice (section 4) | One version per slot, known ten hours after `T`; the 22:00 capture stays as a guard, and a repeated capture adds no version |
| Capacity's `known_at` needs notebook 03's pandas and its own downloads | One query over the notices and the manifest, 0.3 MB, gives notebook 03's inputs for all 33 releases and the twelve dates it printed (sections 6.1, 6.2) | `dwd_capacity_releases` is plain SQL; a copy of the page was taken at `COALESCE(source_time, fetched_at)` |
| "The latest date any copy showed" is final | A later copy moved end of March 2023 by a day, 151 days after it was first listed (section 6.3) | The history rule reads only copies taken before capture start, so the past stays fixed |
| The page's dates need corroborating | Rule dates come a median of 25 days before any copy listing the release, and the files went up on the day (section 6.4) | The rule stands; our daily copy is the evidence from capture start |
| The 16-day margin concerns only the `.xls` releases left out | End of June 2017 is held as an `.xlsx` first linked in 2024 and never announced: known on 23 February 2024, inside the training window (sections 6.1, 6.2) | The trend takes the newest four releases by quarter; by `known_at` it would use the 2017 figure on 385 decision times (section 6.5) |
| The daily capture keeps every version of a release | A re-uploaded file keeps its link, and the capture fetches each file once by name (section 6.6) | Check each file's `Last-Modified` daily and re-fetch on change; until then a correction in place goes unseen |
| Capacity is one DWD model | Its values scan 2,085 MB, its dates 0.3 MB (section 6.7) | Two models: release dates rebuilt freely, area values read incrementally by fetch day |

### Still open

- **The real availability delay of each weather model.** The live capture's values could be matched against archived single runs to tell which run was current at 08:45, which bounds each delay. The archive now ends with the runs known on the morning of 15 November, the day capture started; capture-era days need a periodic re-run of its history
- **How large and how late TEPCO's corrections are.** None so far; the monthly re-capture on 1 December is the first chance to see one
- **The first captured capacity release.** End of June 2026, due by mid-December: it should be `captured` at the fetch that holds both its files, with no rule-dated release moving
- **Corrections to capacity in place.** Whether a re-uploaded release's figures differ, and by how much, stays unknown until the capture watches `Last-Modified`
- **How much fresher the live forecast is than `previous_day2`.** Training sees a forecast about two days old, serving one about a day old; the backtest should measure what that does to the error

## Where the sources and the data disagree

| Where | Says | We found | Handled by |
| --- | --- | --- | --- |
| Phase-1 plan, point-in-time rules | Single Runs: "per slot, the latest run with `known_at ≤ T` that covers the slot" | With the assumed delays the latest run lands exactly at `T` nearly every day, so the rule's outcome hinges on an assumption | The older run until delays are measured (section 3.2) |
| Phase-1 plan, initial load | Previous Runs history "2024-01 to now" | "Now" was the load day: the history stops there, two target days before the live capture can serve | Topped up on 2026-11-16, after capture start; a source per target day and a test for the seam (section 3.4) |
| Notebook 02, section 9 | Previous Runs: later versions "– (an archive, fetched once)" | Fetched twice, lead 0 of the hours just before the first fetch changed | Versioned like every source: a changed value is a new version, known at the later fetch (section 3.4) |
| Notebook 03, what it made known | "The three `.xls` releases of 2017–2018 are left out"; the margin is for "a file announced on no copy" | The `.xls` releases are 2017-09, 2017-12 and 2018-03; end of June 2017 is an `.xlsx`, first linked in 2024 and never announced, so the margin applies to it | Dated by the margin, 23 February 2024; the trend orders by quarter (sections 6.2, 6.5) |
| Notebook 03, where the sources disagree | "From now on the daily capture keeps each version it sees" | It fetches each release file once by name, and a re-upload keeps its link: it never sees a later version | Open: watch each file's `Last-Modified` (section 6.6) |
| The capture of 15 November | Daily captures | Two of the three `openmeteo_live` captures that day were test runs after `T`; they are real evidence but serve no target day | A live capture serves D only if fetched before `T` on D−1 (section 3.3) |

## Sources

| Page | Supports |
| --- | --- |
| [Open-Meteo forecast API](https://open-meteo.com/en/docs) | Radiation as the mean of the preceding hour; GMT by default |
| [Open-Meteo Previous Runs API](https://open-meteo.com/en/docs/previous-runs-api) | Leads as `previous_dayN` variables |
| [Open-Meteo Single Runs API](https://open-meteo.com/en/docs/single-runs-api) | Runs by start time; availability delays |
| [TEPCO area supply and demand](https://www.tepco.co.jp/forecast/html/area_jukyu-j.html) | The monthly actuals, MW averaged per slot |
| [FIT/FIP disclosure site, summary page](https://www.fit-portal.go.jp/PublicInfoSummary) | The notices and release links read in section 6 |
| [BigQuery `UNPIVOT`](https://cloud.google.com/bigquery/docs/reference/standard-sql/query-syntax#unpivot_operator) | `INCLUDE NULLS` |